[Reference](https://blog.cubed.run/i-rewrote-my-fastapi-error-handling-here-is-what-i-found-cba6c73a21be$0)

# Global Exception Handler

## Before

In [ ]:
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

app = FastAPI()
@app.exception_handler(Exception)
async def global_exception_handler(request: Request, exc: Exception):
    return JSONResponse(
        status_code=500,
        content={"detail": "Internal server error"}
    )

## After

In [1]:
import logging
import traceback
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

logger = logging.getLogger(__name__)
app = FastAPI()
@app.exception_handler(Exception)
async def global_exception_handler(request: Request, exc: Exception):
    logger.error(
        "Unhandled exception",
        extra={
            "path": request.url.path,
            "method": request.method,
            "exception_type": type(exc).__name__,
            "exception_message": str(exc),
            "traceback": traceback.format_exc()
        }
    )
    return JSONResponse(
        status_code=500,
        content={"detail": "Internal server error"}
    )

# Custom Exception Classes

## Before

In [3]:
@app.get("/orders/{order_id}")
async def get_order(order_id: int, db: AsyncSession = Depends(get_db)):
    order = await db.get(Order, order_id)
    if not order:
        raise HTTPException(status_code=404, detail="Not found")
    return order

## After:

In [4]:
class ResourceNotFoundError(Exception):
    def __init__(self, resource: str, resource_id: int):
        self.resource = resource
        self.resource_id = resource_id
        super().__init__(f"{resource} with id {resource_id} not found")

@app.exception_handler(ResourceNotFoundError)
async def resource_not_found_handler(request: Request, exc: ResourceNotFoundError):
    logger.warning(
        "Resource not found",
        extra={"resource": exc.resource, "resource_id": exc.resource_id}
    )
    return JSONResponse(
        status_code=404,
        content={
            "detail": f"{exc.resource} not found",
            "resource_id": exc.resource_id
        }
    )
@app.get("/orders/{order_id}")
async def get_order(order_id: int, db: AsyncSession = Depends(get_db)):
    order = await db.get(Order, order_id)
    if not order:
        raise ResourceNotFoundError("Order", order_id)
    return order

# External Service Errors

## Before:

In [5]:
import httpx

async def fetch_user_profile(user_id: int):
    async with httpx.AsyncClient() as client:
        response = await client.get(f"https://profile-service.example.com/users/{user_id}")
        return response.json()

## After:

In [6]:
import httpx
import logging
from tenacity import retry, stop_after_attempt, wait_exponential

logger = logging.getLogger(__name__)
class ExternalServiceError(Exception):
    def __init__(self, service: str, message: str):
        self.service = service
        super().__init__(f"{service} error: {message}")
@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=1, max=10)
)
async def fetch_user_profile(user_id: int):
    try:
        async with httpx.AsyncClient(timeout=5.0) as client:
            response = await client.get(
                f"https://profile-service.example.com/users/{user_id}"
            )
            response.raise_for_status()
            return response.json()
    except httpx.TimeoutException:
        logger.error("Profile service timeout", extra={"user_id": user_id})
        raise ExternalServiceError("ProfileService", "Request timed out")
    except httpx.HTTPStatusError as e:
        logger.error(
            "Profile service error",
            extra={"user_id": user_id, "status_code": e.response.status_code}
        )
        raise ExternalServiceError("ProfileService", f"HTTP {e.response.status_code}")

# Validation Error Responses

## Before:
```
{
  "detail": [
    {
      "loc": ["body", "email"],
      "msg": "value is not a valid email address",
      "type": "value_error.email"
    }
  ]
}
```

## After:

In [7]:
from fastapi.exceptions import RequestValidationError

@app.exception_handler(RequestValidationError)
async def validation_exception_handler(request: Request, exc: RequestValidationError):
    errors = []
    for error in exc.errors():
        field = " -> ".join(str(loc) for loc in error["loc"] if loc != "body")
        errors.append({
            "field": field,
            "message": error["msg"]
        })
    logger.warning(
        "Validation error",
        extra={"path": request.url.path, "errors": errors}
    )
    return JSONResponse(
        status_code=422,
        content={
            "detail": "Validation failed",
            "errors": errors
        }
    )